## 准备数据

In [1]:
import os
import numpy as np
import torch


def mnist_dataset(root='./data'):
    try:
        from torchvision import datasets
    except ImportError as e:
        raise ImportError('Please install torchvision: pip install torchvision') from e

    train_ds = datasets.MNIST(root=root, train=True, download=True)
    test_ds = datasets.MNIST(root=root, train=False, download=True)

    x = train_ds.data.numpy().astype(np.float32) / 255.0
    y = train_ds.targets.numpy().astype(np.int64)
    x_test = test_ds.data.numpy().astype(np.float32) / 255.0
    y_test = test_ds.targets.numpy().astype(np.int64)
    return (x, y), (x_test, y_test)


## Demo numpy based auto differentiation

In [2]:
import numpy as np

class Matmul:
    def __init__(self):
        self.mem = {}
        
    def forward(self, x, W):
        h = np.matmul(x, W)
        self.mem={'x': x, 'W':W}
        return h
    
    def backward(self, grad_y):
        '''
        x: shape(N, d)
        w: shape(d, d')
        grad_y: shape(N, d')
        '''
        x = self.mem['x']
        W = self.mem['W']
        
        ####################
        '''????????????'''
        grad_x = np.matmul(grad_y, W.T)
        grad_W = np.matmul(x.T, grad_y)
        ####################
        return grad_x, grad_W


class Relu:
    def __init__(self):
        self.mem = {}
        
    def forward(self, x):
        self.mem['x']=x
        return np.where(x > 0, x, np.zeros_like(x))
    
    def backward(self, grad_y):
        '''
        grad_y: same shape as x
        '''
        ####################
        '''??relu ?????????'''
        grad_x = grad_y * (self.mem['x'] > 0).astype(grad_y.dtype)
        ####################
        return grad_x
    


class Softmax:
    '''
    softmax over last dimention
    '''
    def __init__(self):
        self.epsilon = 1e-12
        self.mem = {}
        
    def forward(self, x):
        '''
        x: shape(N, c)
        '''
        x_exp = np.exp(x)
        partition = np.sum(x_exp, axis=1, keepdims=True)
        out = x_exp/(partition+self.epsilon)
        
        self.mem['out'] = out
        self.mem['x_exp'] = x_exp
        return out
    
    def backward(self, grad_y):
        '''
        grad_y: same shape as x
        '''
        s = self.mem['out']
        sisj = np.matmul(np.expand_dims(s,axis=2), np.expand_dims(s, axis=1)) # (N, c, c)
        g_y_exp = np.expand_dims(grad_y, axis=1)
        tmp = np.matmul(g_y_exp, sisj) #(N, 1, c)
        tmp = np.squeeze(tmp, axis=1)
        tmp = -tmp+grad_y*s 
        return tmp
    
class Log:
    '''
    softmax over last dimention
    '''
    def __init__(self):
        self.epsilon = 1e-12
        self.mem = {}
        
    def forward(self, x):
        '''
        x: shape(N, c)
        '''
        out = np.log(x+self.epsilon)
        
        self.mem['x'] = x
        return out
    
    def backward(self, grad_y):
        '''
        grad_y: same shape as x
        '''
        x = self.mem['x']
        
        return 1./(x+1e-12) * grad_y


## Gradient check

In [3]:
# Optional autograd sanity-check snippets (PyTorch)\n
# x = np.random.normal(size=[5, 6]).astype(np.float32)\n
# W = np.random.normal(size=[6, 4]).astype(np.float32)\n
# aa = Matmul()\n
# out = aa.forward(x, W)\n
# grad = aa.backward(np.ones_like(out))\n
# print(grad)\n
#\n
# xt = torch.tensor(x, requires_grad=True)\n
# Wt = torch.tensor(W)\n
# y = xt @ Wt\n
# loss = y.sum()\n
# loss.backward()\n
# print(xt.grad)\n

# Final Gradient Check

In [4]:
x = np.random.normal(size=[5, 6]).astype(np.float32)
label = np.zeros_like(x)
label[0, 1] = 1.
label[1, 0] = 1
label[2, 3] = 1
label[3, 5] = 1
label[4, 0] = 1

x = np.random.normal(size=[5, 6])
W1 = np.random.normal(size=[6, 5])
W2 = np.random.normal(size=[5, 6])

mul_h1 = Matmul()
mul_h2 = Matmul()
relu = Relu()
softmax = Softmax()
log = Log()

h1 = mul_h1.forward(x, W1)  # shape(5, 5)
h1_relu = relu.forward(h1)
h2 = mul_h2.forward(h1_relu, W2)
h2_soft = softmax.forward(h2)
h2_log = log.forward(h2_soft)

h2_log_grad = log.backward(label)
h2_soft_grad = softmax.backward(h2_log_grad)
h2_grad, W2_grad = mul_h2.backward(h2_soft_grad)
h1_relu_grad = relu.backward(h2_grad)
h1_grad, W1_grad = mul_h1.backward(h1_relu_grad)

print(h2_log_grad)
print('--' * 20)

# Torch autograd check
xt = torch.tensor(x, dtype=torch.float32)
W1t = torch.tensor(W1, dtype=torch.float32, requires_grad=True)
W2t = torch.tensor(W2, dtype=torch.float32, requires_grad=True)
label_t = torch.tensor(label, dtype=torch.float32)

h1_t = xt @ W1t
h1_relu_t = torch.relu(h1_t)
h2_t = h1_relu_t @ W2t
prob_t = torch.softmax(h2_t, dim=-1)
log_prob_t = torch.log(prob_t)
loss_t = torch.sum(label_t * log_prob_t)

grads_prob = torch.autograd.grad(loss_t, prob_t, retain_graph=False)[0]
print(grads_prob.detach().numpy())


[[  0.          24.37460545   0.           0.           0.
    0.        ]
 [153.93694229   0.           0.           0.           0.
    0.        ]
 [  0.           0.           0.           2.67166371   0.
    0.        ]
 [  0.           0.           0.           0.           0.
    4.32414378]
 [ 16.83664033   0.           0.           0.           0.
    0.        ]]
----------------------------------------
[[  0.         24.374601    0.          0.          0.          0.       ]
 [153.9369      0.          0.          0.          0.          0.       ]
 [  0.          0.          0.          2.6716638   0.          0.       ]
 [  0.          0.          0.          0.          0.          4.324144 ]
 [ 16.836641    0.          0.          0.          0.          0.       ]]


## 建立模型

In [5]:
class myModel:
    def __init__(self):
        
        self.W1 = np.random.normal(size=[28*28+1, 100])
        self.W2 = np.random.normal(size=[100, 10])
        
        self.mul_h1 = Matmul()
        self.mul_h2 = Matmul()
        self.relu = Relu()
        self.softmax = Softmax()
        self.log = Log()
        
        
    def forward(self, x):
        x = x.reshape(-1, 28*28)
        bias = np.ones(shape=[x.shape[0], 1])
        x = np.concatenate([x, bias], axis=1)
        
        self.h1 = self.mul_h1.forward(x, self.W1) # shape(5, 4)
        self.h1_relu = self.relu.forward(self.h1)
        self.h2 = self.mul_h2.forward(self.h1_relu, self.W2)
        self.h2_soft = self.softmax.forward(self.h2)
        self.h2_log = self.log.forward(self.h2_soft)
            
    def backward(self, label):
        self.h2_log_grad = self.log.backward(-label)
        self.h2_soft_grad = self.softmax.backward(self.h2_log_grad)
        self.h2_grad, self.W2_grad = self.mul_h2.backward(self.h2_soft_grad)
        self.h1_relu_grad = self.relu.backward(self.h2_grad)
        self.h1_grad, self.W1_grad = self.mul_h1.backward(self.h1_relu_grad)
        
model = myModel()


## 计算 loss

In [6]:
def compute_loss(log_prob, labels):
     return np.mean(np.sum(-log_prob*labels, axis=1))
    

def compute_accuracy(log_prob, labels):
    predictions = np.argmax(log_prob, axis=1)
    truth = np.argmax(labels, axis=1)
    return np.mean(predictions==truth)

def train_one_step(model, x, y):
    model.forward(x)
    model.backward(y)
    model.W1 -= 1e-5* model.W1_grad
    model.W2 -= 1e-5* model.W2_grad
    loss = compute_loss(model.h2_log, y)
    accuracy = compute_accuracy(model.h2_log, y)
    return loss, accuracy

def test(model, x, y):
    model.forward(x)
    loss = compute_loss(model.h2_log, y)
    accuracy = compute_accuracy(model.h2_log, y)
    return loss, accuracy

## 实际训练

In [7]:
train_data, test_data = mnist_dataset()
train_label = np.zeros(shape=[train_data[0].shape[0], 10])
test_label = np.zeros(shape=[test_data[0].shape[0], 10])
train_label[np.arange(train_data[0].shape[0]), np.array(train_data[1])] = 1.
test_label[np.arange(test_data[0].shape[0]), np.array(test_data[1])] = 1.

for epoch in range(50):
    loss, accuracy = train_one_step(model, train_data[0], train_label)
    print('epoch', epoch, ': loss', loss, '; accuracy', accuracy)
loss, accuracy = test(model, test_data[0], test_label)

print('test loss', loss, '; accuracy', accuracy)

100%|██████████| 9.91M/9.91M [00:15<00:00, 643kB/s] 
100%|██████████| 28.9k/28.9k [00:00<00:00, 98.6kB/s]
100%|██████████| 1.65M/1.65M [00:02<00:00, 797kB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 1.49MB/s]


epoch 0 : loss 22.909853905612174 ; accuracy 0.13236666666666666
epoch 1 : loss 21.621373022203745 ; accuracy 0.17133333333333334
epoch 2 : loss 19.938834696421203 ; accuracy 0.212
epoch 3 : loss 18.252294765210117 ; accuracy 0.27685
epoch 4 : loss 16.86499508707488 ; accuracy 0.3237833333333333
epoch 5 : loss 14.99213766515467 ; accuracy 0.39405
epoch 6 : loss 14.523971857289649 ; accuracy 0.4201666666666667
epoch 7 : loss 13.948505191755284 ; accuracy 0.43678333333333336
epoch 8 : loss 13.449466943155308 ; accuracy 0.45718333333333333
epoch 9 : loss 12.85472657824425 ; accuracy 0.47583333333333333
epoch 10 : loss 12.342156198126723 ; accuracy 0.49351666666666666
epoch 11 : loss 11.820933753665514 ; accuracy 0.51045
epoch 12 : loss 11.362973037736865 ; accuracy 0.52745
epoch 13 : loss 10.93598470806885 ; accuracy 0.5394166666666667
epoch 14 : loss 10.627953050609818 ; accuracy 0.5544
epoch 15 : loss 10.70841396798198 ; accuracy 0.5487
epoch 16 : loss 11.071193594099194 ; accuracy 0.54